<a href="https://colab.research.google.com/github/Neha-28-fluff/galaxy-vision-ai/blob/main/notebooks/day_2_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
ROOT = Path("/content/drive/MyDrive/galaxy-vision-ai")

PROCESSED = ROOT / "data" / "processed"

# IMAGE_DIR = PROCESSED / "images"

train_df = pd.read_csv(PROCESSED / "train.csv")
val_df = pd.read_csv(PROCESSED / "val.csv")
test_df = pd.read_csv(PROCESSED / "test.csv")

!cp -r "/content/drive/MyDrive/galaxy-vision-ai/data/processed/images" "/content/"
!ls /content/images | head

IMAGE_DIR = Path("/content/images")

100053.jpg
100090.jpg
100122.jpg
100123.jpg
100134.jpg
100143.jpg
100263.jpg
100295.jpg
100322.jpg
100380.jpg


In [ ]:
#  check
train_df.head()

,GalaxyID,Morphology,label,Confidence
0,225927,EdgeOn_Disk,1,0.626851
1,607435,Unbarred_Spiral,5,0.469441
2,407594,Intermediate_Elliptical,2,0.726658
3,683796,Irregular_Merger,3,0.546769
4,113655,Unbarred_Spiral,5,0.733071


In [ ]:
# CNNs require fixed image size.

transform = transforms.Compose([

    transforms.Resize((224,224)),   # 224x224

    transforms.ToTensor(),  # 0-255 → 0-1

    transforms.Normalize(           # Stable Training
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
class GalaxyDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        galaxy_id = int(row["GalaxyID"])

        img_path = (
            self.image_dir /
            f"{galaxy_id}.jpg"
        )

        image = Image.open(
            img_path
        ).convert("RGB")

        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# dataet objects

train_dataset = GalaxyDataset(
    train_df,
    IMAGE_DIR,
    transform
)

val_dataset = GalaxyDataset(
    val_df,
    IMAGE_DIR,
    transform
)

test_dataset = GalaxyDataset(
    test_df,
    IMAGE_DIR,
    transform
)

In [ ]:
# dataset loaders

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# Verify Shapes

images, labels = next(
    iter(train_loader)
)

print(images.shape)

print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [ ]:
NUM_CLASSES = train_df["label"].nunique()

class SimpleCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),

            nn.Linear(
                128,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                NUM_CLASSES
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = SimpleCNN().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
# check
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(next(model.parameters()).device)

True
Tesla T4
cuda:0


In [ ]:
#  training
import time

EPOCHS = 5

for epoch in range(EPOCHS):

    start = time.time()

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = (
        running_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
        f" | Loss: {epoch_loss:.4f}"
        f" | Time: {time.time()-start:.2f}s"
    )


Epoch 1/5 | Loss: 0.8431 | Time: 80.30s
Epoch 2/5 | Loss: 0.8207 | Time: 78.68s
Epoch 3/5 | Loss: 0.7995 | Time: 77.80s
Epoch 4/5 | Loss: 0.7717 | Time: 77.61s
Epoch 5/5 | Loss: 0.7486 | Time: 77.89s


In [ ]:
# from PIL import Image
# import time

# row = train_df.iloc[0]

# img_path = IMAGE_DIR / f"{int(row['GalaxyID'])}.jpg"

# start = time.time()

# img = Image.open(img_path).convert("RGB")

# end = time.time()

# print(end - start)

0.0028901100158691406


In [ ]:
from sklearn.metrics import f1_score

model.eval()

correct = 0
total = 0

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        preds = outputs.argmax(1)

        # Accuracy
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # F1 Score
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy = correct / total

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Macro F1 Score: {macro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")

Validation Accuracy: 0.7340
Macro F1 Score: 0.6209
Weighted F1 Score: 0.7130


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.39      0.05      0.09       171
           1       0.89      0.95      0.91       438
           2       0.72      0.76      0.74       624
           3       0.56      0.39      0.46       228
           4       0.86      0.74      0.79       673
           5       0.61      0.90      0.72       494

    accuracy                           0.73      2628
   macro avg       0.67      0.63      0.62      2628
weighted avg       0.73      0.73      0.71      2628



In [ ]:
MODEL_DIR = ROOT / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epochs": EPOCHS,
    "num_classes": NUM_CLASSES
}

torch.save(
    checkpoint,
    MODEL_DIR / "cnn_baseline_checkpoint.pth"
)

!ls "/content/drive/MyDrive/galaxy-vision-ai/models"

cnn_baseline_checkpoint.pth
